# 🇪🇹 Amharic Speech-to-Speech RAG API Server
### Dual-GPU FastAPI Server + WebSockets + Cloudflare Public Tunnel

This notebook deploys a production-grade **Amharic Speech-to-Speech RAG Assistant** for the **Ethiopian Labour Proclamation No. 1156/2019**.

```
┌────────────────────────────────┐       Public WSS Stream       ┌────────────────────────────────┐
│         CLIENT / WEB UI        │◄─────────────────────────────►│       KAGGLE / COLAB GPU       │
│  • Amharic Voice Recorder      │    Cloudflare Tunnel (Live)   │  • FastAPI Backend (:8000)     │
│  • Sentence Audio Stream       │                               │  • GPU 0: ASR + RAG + TTS      │
│  • Latency Waterfall Telemetry │                               │  • GPU 1: llama-server Gemma 4B│
└────────────────────────────────┘                               └────────────────────────────────┘
```

**Key Highlights:**
- 🎙️ **Hohe ASR**: In-memory mono 16kHz speech transcription with zero disk I/O.
- 🧠 **Hybrid E5 RAG**: Multilingual-E5-Base dense vectors + Character n-gram TF-IDF matching.
- ⚡ **Gemma 4B (llama.cpp)**: Dedicated GPU 1 isolation with real-time SSE phrase-boundary streaming.
- 🔊 **OmniVoice TTS**: 8-step first phrase synthesis for **TTFS < 1.0s** + pre-cached direct responses.
- 🌐 **FastAPI + WebSockets**: Full-duplex `/ws/s2s` streaming + Swagger `/docs` + Glassmorphic Web UI.


## 1. Environment & Dependencies Setup
Install the lightweight server stack and audio dependencies.

In [1]:
# Compatible stack for OmniVoice 0.2.1 + E5 retrieval:
# - omnivoice needs transformers>=5.3.0 (HiggsAudioV2TokenizerModel)
# - sentence-transformers 5.7.x supports transformers v5
# NEVER re-pin transformers to 4.x — that breaks OmniVoice imports.
!pip install -q omnivoice==0.2.1
!pip install -q "sentence-transformers==5.7.0"
!pip install -q "transformers>=5.3.0,<6.0.0"
!pip install -q accelerate sentencepiece librosa soundfile pydub tensorboardX webdataset pandas faiss-cpu scikit-learn pymupdf requests "huggingface_hub>=0.34.0" fastapi "uvicorn[standard]" websockets pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 4.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 85.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 75.1 MB/s eta 0:00:00:00:0100:01


## 1b. Dependency Version Gate
Fail fast unless OmniVoice can import (needs `transformers>=5.3.0`) and Sentence-Transformers is pinned to the v5-compatible `5.7.0` release.


In [2]:
# Shared package gate when amharic_rag is on sys.path; inline fallback otherwise.
# Also verifies OmniVoice can see HiggsAudioV2TokenizerModel (transformers>=5.3).
try:
    from server.deps_check import (
        assert_dependency_versions,
        assert_omnivoice_transformers_symbols,
        REQUIRED_VERSIONS,
        installed_version,
    )
    print("Using server.deps_check:", REQUIRED_VERSIONS)
    print("Installed transformers:", installed_version("transformers"))
    assert_dependency_versions(allow_missing=False)
    assert_omnivoice_transformers_symbols()
    print("Dependency gate passed (versions + OmniVoice import symbols).")
except ImportError:
    import importlib.metadata as md

    def _ver(pkg):
        try:
            return md.version(pkg)
        except md.PackageNotFoundError:
            return None

    def _parse(v):
        parts = []
        for chunk in (v or "").split("."):
            d = "".join(ch for ch in chunk if ch.isdigit())
            if not d:
                break
            parts.append(int(d))
        return tuple(parts) if parts else (0,)

    exact = {"omnivoice": "0.2.1", "sentence-transformers": "5.7.0"}
    mismatches = []
    for pkg, expected in exact.items():
        actual = _ver(pkg) or "NOT INSTALLED"
        ok = actual == expected
        print(f"{'OK' if ok else 'FAIL'}  {pkg}: {actual} (expected {expected})")
        if not ok:
            mismatches.append((pkg, actual, expected))

    tf = _ver("transformers")
    tf_ok = tf is not None and _parse(tf) >= (5, 3, 0) and _parse(tf) < (6, 0, 0)
    print(f"{'OK' if tf_ok else 'FAIL'}  transformers: {tf} (expected >=5.3.0,<6.0.0)")
    if not tf_ok:
        mismatches.append(("transformers", tf or "NOT INSTALLED", ">=5.3.0,<6.0.0"))

    if mismatches:
        raise RuntimeError(
            "Incompatible dependency versions. Re-run install cell. "
            "Need omnivoice==0.2.1, sentence-transformers==5.7.0, transformers>=5.3.0. "
            f"Mismatches: {mismatches}"
        )

    try:
        from transformers import HiggsAudioV2TokenizerModel  # noqa: F401
        import omnivoice  # noqa: F401
        from omnivoice import OmniVoice  # noqa: F401
    except Exception as e:
        raise RuntimeError(
            "OmniVoice/transformers symbols missing. Install transformers>=5.3.0 "
            "(do not pin transformers 4.x)."
        ) from e
    print("Dependency gate passed (inline fallback + OmniVoice symbols).")


OK  omnivoice: 0.2.1 (expected 0.2.1)
OK  sentence-transformers: 5.7.0 (expected 5.7.0)
OK  transformers: 5.16.1 (expected >=5.3.0,<6.0.0)
Dependency gate passed (inline fallback + OmniVoice symbols).


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


## 2. Hardware Topology & Multi-GPU Detection
Detect available GPUs and configure hardware isolation across CUDA devices.

In [3]:
import os, sys, shutil, torch
from pathlib import Path

print('PyTorch Version:', torch.__version__)
print('CUDA Available: ', torch.cuda.is_available())
num_gpus = torch.cuda.device_count()
print('GPUs Detected:  ', num_gpus)
for i in range(num_gpus):
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)')

if num_gpus >= 2:
    print('\n🔥 KAGGLE DUAL-GPU MODE ENABLED:')
    print('   • GPU 0 (cuda:0): ASR (Hohe) + Hybrid RAG (E5) + TTS (OmniVoice)')
    print('   • GPU 1 (cuda:1): Dedicated LLM Server (Gemma 4B via llama-server)')
else:
    print('\n⚡ SINGLE-GPU MODE (Colab / T4):')
    print('   • GPU 0 (cuda:0): All 4 engines running concurrently (~10.1 GB VRAM)')


PyTorch Version: 2.11.0+cu128
CUDA Available:  True
GPUs Detected:   2
  • GPU 0: Tesla T4 (15.6 GB VRAM)
  • GPU 1: Tesla T4 (15.6 GB VRAM)

🔥 KAGGLE DUAL-GPU MODE ENABLED:
   • GPU 0 (cuda:0): ASR (Hohe) + Hybrid RAG (E5) + TTS (OmniVoice)
   • GPU 1 (cuda:1): Dedicated LLM Server (Gemma 4B via llama-server)


## 3. Project Workspace Setup
Mounts the modular `amharic_rag` package and ensures executable permissions on `llama-server`.

In [4]:
import os, sys, shutil, glob
from pathlib import Path

# Newer Kaggle mounts datasets at /kaggle/input/datasets/<owner>/<slug>/..., older ones at /kaggle/input/<slug>/...
# so no fixed path list works. Search for the `server` package instead (it is what the later cells import).
def find_project():
    work = Path('/kaggle/working/amharic_rag')
    if (work / 'server' / 'launcher.py').exists():
        return work
    for base in ['/kaggle/input', '/content', '.']:
        hits = sorted(glob.glob(f'{base}/**/server/launcher.py', recursive=True), key=len)
        if hits:
            return Path(hits[0]).parent.parent.resolve()
    return None

project_root = find_project()

if project_root is None:
    print('Nothing under /kaggle/input contains server/launcher.py. What IS attached:')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root[len('/kaggle/input'):].count(os.sep)
        if depth <= 3:
            print('  ' * depth + os.path.basename(root) + '/', f'({len(files)} files)')
        else:
            dirs[:] = []
    raise AssertionError(
        'Project not found. In the right-hand panel use Add Input -> Datasets and attach '
        '"samuelkeno/amharic-speech-to-speech-rag-api" (it must be listed under Input), then re-run. '
        'If it IS attached, the tree above shows where the server/ folder really is.')

# Kaggle inputs are read-only: copy to the writable folder (dirs_exist_ok so a half-finished earlier copy is completed)
if str(project_root).startswith('/kaggle/input'):
    dest = Path('/kaggle/working/amharic_rag')
    print(f'Copying {project_root} -> {dest} ...')
    shutil.copytree(str(project_root), str(dest), dirs_exist_ok=True)
    project_root = dest

print('Using Project Root:', project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Report (not fail) on the other things the pipeline expects
for rel in ['server/config.py', 'src/normalize.py', 'index/chunks.faiss', 'index/chunks.json', 'data/questions_mapped.json']:
    print(('  ok       ' if (project_root / rel).exists() else '  MISSING  ') + rel)

# llama-server must be executable
for bin_cand in [project_root / 'bin' / 'llama-server-sm75', project_root / 'bin' / 'llama-server']:
    if bin_cand.exists():
        os.chmod(str(bin_cand), 0o755)
        print('Set executable permissions on', bin_cand)


Copying /kaggle/input/datasets/samuelkeno/amharic-speech-to-speech-rag-api/amharic_rag -> /kaggle/working/amharic_rag ...
Using Project Root: /kaggle/working/amharic_rag
  ok       server/config.py
  ok       src/normalize.py
  ok       index/chunks.faiss
  ok       index/chunks.json
  ok       data/questions_mapped.json
Set executable permissions on /kaggle/working/amharic_rag/bin/llama-server-sm75


## 3b. Runtime preparation (HF token, llama-server CUDA libraries, tunnel)
These three fixes come from the first notebook: an exported `HF_TOKEN`, the CUDA 13 runtime the prebuilt `llama-server` needs on Kaggle's CUDA 12 image, and a Cloudflare tunnel that works without UDP.

In [5]:
import os, glob, subprocess, sys, re

# --- 1. Hugging Face token (exported so every download in the server process is authenticated) ---
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or None
except Exception as e:
    print(f"Kaggle secret HF_TOKEN not available ({type(e).__name__}).")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("Using HF token")
else:
    print("No HF_TOKEN: continuing anonymously.")

# --- 2. llama-server needs libcudart.so.13 / libcublas.so.13 (Kaggle ships CUDA 12) ---
CUDA13_DIR = "/kaggle/working/cuda13_libs"
def _add_lib_dir(d):
    cur = os.environ.get("LD_LIBRARY_PATH", "")
    if d not in cur.split(":"):
        os.environ["LD_LIBRARY_PATH"] = d + (":" + cur if cur else "")
def _missing(path):
    r = subprocess.run(["ldd", path], capture_output=True, text=True, env=os.environ)
    return [l.split("=>")[0].strip() for l in r.stdout.splitlines() if "not found" in l]

for d in glob.glob(f"{CUDA13_DIR}/nvidia/*/lib"):
    _add_lib_dir(d)
for b in [project_root / "bin" / "llama-server-sm75", project_root / "bin" / "llama-server"]:
    if not b.exists():
        continue
    miss = _missing(str(b))
    if any(m.endswith(".13") for m in miss):
        print("llama-server needs", miss, "-> installing CUDA 13 runtime libraries ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", CUDA13_DIR,
                        "nvidia-cuda-runtime", "nvidia-cublas"], check=True)
        for d in glob.glob(f"{CUDA13_DIR}/nvidia/*/lib"):
            _add_lib_dir(d)
    print(b.name, "-> still missing:", _missing(str(b)) or "nothing, OK")
    break

# --- 3. Cloudflare tunnel: Kaggle blocks UDP, so the default QUIC link prints a URL that then shows "Error 1033".
#        Force http2 over IPv4 in the launcher (best effort: the file is edited only if the pattern is found once).
launcher = project_root / "server" / "launcher.py"
txt = launcher.read_text(encoding="utf-8")
if "cloudflared" in txt and "--protocol" not in txt:
    new, n = re.subn(r"""(["']tunnel["'])\s*,""", r'\1, "--protocol", "http2", "--edge-ip-version", "4",', txt, count=1)
    if n:
        launcher.write_text(new, encoding="utf-8")
        print("launcher.py patched to use http2:", [l.strip() for l in new.splitlines() if "--protocol" in l][0])
    else:
        print("Could not auto-patch launcher.py; add '--protocol','http2','--edge-ip-version','4' after 'tunnel' in its cloudflared command.")
else:
    print("launcher.py: cloudflared command already pinned or not used.")

Using HF token
llama-server needs ['libcudart.so.13', 'libcublas.so.13'] -> installing CUDA 13 runtime libraries ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.3/439.3 MB 4.1 MB/s eta 0:00:00
llama-server-sm75 -> still missing: nothing, OK
launcher.py: cloudflared command already pinned or not used.


## 4. Run Modular Verification Tests
Runs unit tests across the server components before launching the live server.

In [6]:
from server.config import NUM_GPUS, MULTI_GPU, LLM_GPU_ID
from server.llm_service import sanitize_to_ethiopic, split_ready
from server.retrieval_service import _minmax
from server.orchestrator import DIRECT_INTENTS

print('--- Modular Server Verification ---')
print(f'• Multi-GPU routing: LLM assigned to GPU {LLM_GPU_ID}')
print(f'• Direct conversational intents: {DIRECT_INTENTS}')

# Test phrase boundary chunking
test_buf = 'የሙከራ ጊዜ ስልሳ ቀናት ነው። ሁለተኛው ድንጋጌ ደግሞ ይህ ነው።'
phrases, rest, _ = split_ready(test_buf, first=True)
print(f'• Sentence boundary test: Extracted {len(phrases)} phrases: {phrases}')
print('✅ Modular components verified successfully.')


--- Modular Server Verification ---
• Multi-GPU routing: LLM assigned to GPU 1
• Direct conversational intents: {'GOODBYE', 'GREETING', 'META_IDENTITY', 'GREETING_SMALLTALK', 'THANK_YOU'}
• Sentence boundary test: Extracted 2 phrases: ['የሙከራ ጊዜ ስልሳ ቀናት ነው።', ' ሁለተኛው ድንጋጌ ደግሞ ይህ ነው።']
✅ Modular components verified successfully.


## 5. Launch Live FastAPI Server + Public Cloudflare Tunnel
Starts the FastAPI server with WebSocket streaming, loads models into GPU VRAM, and opens a secure public Cloudflare tunnel.

In [ ]:
import threading, time, subprocess, traceback, requests
from server.launcher import run_server

PORT = 8000
KEEP_ALIVE = True        # True: this cell stays running so the link keeps working (interrupt it to stop waiting)

# Leftovers from an earlier attempt: the previous run opened a tunnel and then crashed, so kill stray processes.
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
subprocess.run(["pkill", "-9", "-f", "llama-server"], check=False)
time.sleep(1)

# Why the old cell crashed: run_server() calls uvicorn.run(), which calls asyncio.run(), and a notebook cell already runs
# inside an event loop ("asyncio.run() cannot be called from a running event loop"). In its own thread there is no
# running loop, so uvicorn starts normally.
_err = []
def _serve():
    try:
        run_server(port=PORT, tunnel="cloudflare")
    except BaseException:
        _err.append(traceback.format_exc())

srv_thread = threading.Thread(target=_serve, name="s2s-server", daemon=True)
srv_thread.start()

# The tunnel link is printed BEFORE the models are loaded, so wait until the server really answers.
t0, ready, last = time.time(), False, 0.0
while time.time() - t0 < 1800:
    if not srv_thread.is_alive():
        print("\nServer thread stopped:\n", _err[0] if _err else "(no error captured)")
        break
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3)
        if r.status_code == 200:
            ready = True
            print(f"\n✅ Server is READY after {time.time() - t0:.0f}s. /health -> {r.text[:300]}")
            print("Use the https://...trycloudflare.com link printed above.")
            break
    except requests.RequestException:
        pass
    if time.time() - last > 30:
        print(f"  ... still loading models / downloading ({time.time() - t0:.0f}s)")
        last = time.time()
    time.sleep(2)
else:
    print("Not ready after 30 min. Check the output above.")

if ready and KEEP_ALIVE:
    print("Server running. Leave this cell running while you use it. (Interrupting only stops waiting; "
          "restart the kernel to stop the server.)")
    try:
        while srv_thread.is_alive():
            time.sleep(5)
    except KeyboardInterrupt:
        pass


[Launcher] Downloading cloudflared tunnel client...
  ... still loading models / downloading (0s)
[Launcher] Opening Cloudflare Tunnel to localhost:8000...

  🚀 AMHARIC SPEECH-TO-SPEECH RAG API IS LIVE!
  🌐 Interactive Web UI:  https://metals-shared-hosting-assistance.trycloudflare.com
  📖 Swagger UI Docs:     https://metals-shared-hosting-assistance.trycloudflare.com/docs
  🎙️ Real-time WebSocket: wss://metals-shared-hosting-assistance.trycloudflare.com/ws/s2s
  🏥 Health Check:        https://metals-shared-hosting-assistance.trycloudflare.com/health



INFO:     Started server process [49]
INFO:     Waiting for application startup.


[FastAPI] Initializing Speech-to-Speech Pipeline Models...

  🚀 INITIALIZING AMHARIC SPEECH-TO-SPEECH RAG PIPELINE 
[ASRService] Loading snapwre/hohe-asr-amharic onto cuda:0 (torch.float16)...


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

... (33 outputs left out)

INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


[TTSService] Ready. Cached 12 router clips. Voice lock: yes.
[Orchestrator] Warmed answer cache with 12 FAQ entries.
  ✅ ALL 4 ENGINES LOADED & READY FOR LIVE TRAFFIC     

INFO:     127.0.0.1:36086 - "GET /health HTTP/1.1" 200 OK

✅ Server is READY after 129s. /health -> {"status":"healthy","num_gpus":2,"multi_gpu":true,"gpu_topology":{"asr_device":"cuda:0","retrieval_device":"cuda:0","tts_device":"cuda:0","llm_gpu_id":1,"llm_port":8081},"models_ready":{"asr":true,"retrieval":true,"llm":true,"tts":true},"vram_allocated_gb":4.37}
Use the https://...trycloudflare.com link printed above.
Server running. Leave this cell running while you use it. (Interrupting only stops waiting; restart the kernel to stop the server.)
INFO:     102.213.71.230:0 - "GET / HTTP/1.1" 200 OK


INFO:     102.213.71.230:0 - "WebSocket /ws/s2s" [accepted]
INFO:     connection open


[WebSocket] Client connected.
INFO:     102.213.71.230:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     102.213.71.230:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     102.213.71.230:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     102.213.71.230:0 - "GET /openapi.json HTTP/1.1" 200 OK


## 6. How to Test & Query Your Live API
You can test the API from your browser or any Python client:

### A. Browser Testing
1. Open the **Interactive Web UI** link printed above (e.g. `https://xxxx.trycloudflare.com`).
2. Click the glowing **🎙️ Microphone** button and speak in Amharic (or type in the text box).
3. Listen to the real-time sentence-by-sentence streaming audio response and inspect the latency waterfall.

### B. Interactive Swagger Docs
1. Open `https://xxxx.trycloudflare.com/docs`.
2. Test `POST /api/v1/chat`, `POST /api/v1/retrieve`, and `GET /health`.

### C. WebSocket Python Client Example
```python
import asyncio, websockets, json

async def query_s2s():
    uri = 'wss://YOUR_CLOUDFLARE_URL.trycloudflare.com/ws/s2s'
    async with websockets.connect(uri) as ws:
        # Send text question
        await ws.send(json.dumps({'type': 'text', 'text': 'የሙከራ ጊዜ ስንት ቀን ነው?'}))
        
        while True:
            msg = json.loads(await ws.recv())
            if msg['type'] == 'transcript':
                print('User:', msg['text'])
            elif msg['type'] == 'audio':
                print('Assistant Sentence:', msg['sentence'])
            elif msg['type'] == 'done':
                print('Completed in:', msg['total_compute_s'], 's')
                break

asyncio.run(query_s2s())
```
